# <center>Correlated equilibria</center>
### <center>Alfred Galichon (NYU & Sciences Po) and Antoine Jacquet (Sciences Po)</center>
## <center>'math+econ+code' masterclass series</center>
#### <center>With python code examples</center>
© 2018–2025 by Alfred Galichon. 
Past and present support from NSF grant DMS-1716489, ERC grant CoG-866274 are acknowledged, as well as inputs from contributors listed [here](http://www.math-econ-code.org/team).

**If you reuse material from this masterclass, please cite as:**<br>
Alfred Galichon, 'math+econ+code' masterclass series. https://www.math-econ-code.org/


### References

* Aumann (1974). "Subjectivity and correlation in randomized strategies". *Journal of Mathematical Economics*.
* Forges, F. (1990). "Correlated Equilibrium in Two-Person Zero-Sum Games". *Econometrica*.

### Learning objectives

* Correlated equilibrium and linear programming formulation
* Computation using Gurobi and simplex algorithm


In [1]:
#!pip install mec --upgrade
import numpy as np
import gurobipy as grb


Game of chicken?

# Correlated equilibrium in a two-player bimatrix game

We consider the same set-up as in the lecture on Nash equilibria: a two-player game where player 1 plays $i \in \mathcal I$, player 2 plays $j \in \mathcal J$, and their payoffs are respectively $A_{ij}$ and $B_{ij}$.

A *correlated equilibrium* is a vector of probabilities $(\pi_{ij})_{i \in \mathcal I, j \in \mathcal J}$ such that $\sum_{ij} \pi_{ij} = 1$ and

\begin{align}
\textstyle\sum_j \pi_{ij} A_{ij} &\geq \textstyle\sum_j \pi_{ij} A_{i' j} \quad \text{for all $i, i'$},  \\
\textstyle\sum_i \pi_{ij} B_{ij} &\geq \textstyle\sum_i \pi_{ij} B_{i j'} \quad \text{for all $j, j'$}.
\end{align}

This can be interpreted as follows: with probability $\pi_{ij}$, a mediator sends to player 1 a private signal instructing him to play $i$, and to player 2 a private signal instructing her to play $j$.
Then $(\pi_{ij})_{i \in \mathcal I, j \in \mathcal J}$ is a correlated equilibrium if no player has an ex-ante incentive to deviate from the instruction they received, assuming that the other player is going to follow it.

*Remark.* Considering $\pi$ as an $\mathcal I \times \mathcal J$ matrix, the equilibrium conditions can be rewritten:

$(A \pi^\top)_{ii} \geq (A \pi^\top)_{i' i}$ for all $i, i'$  
$(B^\top \pi)_{jj} \geq (B^\top \pi)_{j' j}$ for all $j, j'$.

*Exercise.* Show that if $(p,q)$ is a Nash equilibrium, then the vector $(\pi_{ij})$ defined by $\pi_{ij} = p_i q_j$ is a correlated equilibrium.

We import the `Bimatrix_game` class that we built to study Nash equilibria.
(We use again the data from Palacios–Huerta on penalty kicks.)

We then build a method `is_Aumann_eq` to check whether a vector $(\pi_{ij})$ is a correlated equilibrium, and we test it on a Nash equilibrium.

In [2]:
from mec.gt import Bimatrix_game

penalty_nonzero_sum = Bimatrix_game.create_penalty_game(zero_sum=False)
print('A_i_j =\n', penalty_nonzero_sum.A_i_j)
print('B_i_j =\n', penalty_nonzero_sum.B_i_j)

A_i_j =
 [[ 53.21  71.35  93.8 ]
 [ 90.26  42.81  86.12]
 [ 96.88 100.    75.43]]
B_i_j =
 [[ 96.79  28.65   6.2 ]
 [  9.74 107.19  13.88]
 [  3.12   0.    74.57]]


In [3]:
def Bimatrix_game_is_AumannEq(self, π_i_j, tol=1e-5):
    if abs(π_i_j.sum() - 1) > tol :
        print('π_i_j does not sum to 1.')
        return False
    for i in range(self.nbi) :
        for k in range(self.nbi) :
            if self.A_i_j[i,:] @ π_i_j[i,:] < self.A_i_j[k,:] @ π_i_j[i,:] - tol :
                print('Player 1 can deviate from i = ' + str(i) + ' with k = ' + str(k) + '.')
                return False
    for j in range(self.nbj) :
        for l in range(self.nbj) :
            if self.B_i_j[:,j] @ π_i_j[:,j] < self.B_i_j[:,l] @ π_i_j[:,j] - tol :
                print('Player 2 can deviate from j = ' + str(j) + ' with l = ' + str(l) + '.')
                return False
    return True

Bimatrix_game.is_AumannEq = Bimatrix_game_is_AumannEq

In [4]:
Nash_sol = penalty_nonzero_sum.lemke_howson_solve()
p_i, q_j = Nash_sol['p_i'], Nash_sol['q_j']

penalty_nonzero_sum.is_AumannEq(np.outer(p_i, q_j))

True

## Linear programming formulation

Let's write in matrix form the two inequalities which define the correlated equilibrium.

The matrix corresponding to the first inequality is

\begin{equation}
    \left[ \begin{smallmatrix}
    A_{11}-A_{11} & A_{12}-A_{12} & \dots & A_{1J}-A_{1J} & & & & & & & & & \\
    A_{11}-A_{21} & A_{12}-A_{22} & \dots & A_{1J}-A_{2J} & & & & & & & & & \\
    \vdots        & \vdots        &       & \vdots        & & & & & & & & & \\
    A_{11}-A_{I1} & A_{12}-A_{I2} & \dots & A_{1J}-A_{IJ} & & & & & & & & & \\
    & & & & A_{21}-A_{11} & A_{22}-A_{12} & \dots & A_{2J}-A_{1J} & & & & & \\
    & & & & A_{21}-A_{21} & A_{22}-A_{22} & \dots & A_{2J}-A_{2J} & & & & & \\
    & & & & \vdots        & \vdots        &       & \vdots        & & & & & \\
    & & & & A_{21}-A_{I1} & A_{22}-A_{I2} & \dots & A_{2J}-A_{IJ} & & & & & \\
    & & & &               &               &       &               & \ddots & & & & \\
    & & & & & & & & & A_{I1}-A_{11} & A_{I2}-A_{12} & \dots & A_{IJ}-A_{1J} \\
    & & & & & & & & & A_{I1}-A_{21} & A_{I2}-A_{22} & \dots & A_{IJ}-A_{2J} \\
    & & & & & & & & & \vdots        & \vdots        &       & \vdots        \\
    & & & & & & & & & A_{I1}-A_{I1} & A_{I2}-A_{I2} & \dots & A_{IJ}-A_{IJ} \\
    \end{smallmatrix} \right].
\end{equation}

Using Kronecker products, we are able to rewrite it as

\begin{equation}
M_A = \big[ (\text I_{\mathcal I} \otimes 1_{\mathcal J}^\top) \, \text{diag}(A) \big] \otimes 1_{\mathcal I} - \text I_{\mathcal I} \otimes A
\end{equation}

where $\text{diag}(A)$ is the diagonal matrix of size $IJ \times IJ$ obtained by lining the entries of the matrix $A$ along the diagonal (in row-major order).

- Alternative with Hadamard product:
\begin{equation}
M_A = \big[ (\text I_{\mathcal I} \otimes 1_{\mathcal J}^\top) \odot ( 1_{\mathcal I}^\top \otimes A) \big] \otimes 1_{\mathcal I} - \text I_{\mathcal I} \otimes A
\end{equation}

- Alternative wih vectorization:  
Let $M_1 = I_{\mathcal I} \otimes 1_{\mathcal I} \otimes I_{\mathcal J}$,  
$M_2 = 1_{\mathcal I} \otimes I_{\mathcal I}  \otimes I_{\mathcal J}$,  
so that $\sum_{i, i', j} \pi_{ij} (A_{ij} - A_{i' j}) = (M_1 \text{vec}\, \pi)^\top (M_1 - M_2)\text{vec}\, A$.  
We want the sum to be only on $j$, for all $i, i'$ fixed:
\begin{equation}
\sum_j \pi_{ij} (A_{ij} - A_{i' j}) = (M_1 \text{vec}\, \pi)^\top (S_i \otimes S_{i'} \otimes I_{\mathcal J}) (M_1 - M_2)\text{vec}\, A
\end{equation}
where $S_i$ is the matrix with 0 everywhere except at index $ii$, where it is 1.


The matrix corresponding to the second inequality is

\begin{equation}
    \left[ \begin{smallmatrix}
    B_{11}-B_{11} &               &        &               &
    B_{21}-B_{21} &               &        &               &       &
    B_{I1}-B_{I1} &               &        &                \\
    B_{11}-B_{12} &               &        &               &
    B_{21}-B_{22} &               &        &               &       &
    B_{I1}-B_{I2} &               &        &                \\
    \vdots        &               &        &               &
    \vdots        &               &        &               & \dots &
    \vdots        &               &        &                \\
    B_{11}-B_{1J} &               &        &               &
    B_{21}-B_{2J} &               &        &               &       &
    B_{I1}-B_{IJ} &               &        &                \\
                  & B_{12}-B_{11} &        &               &
                  & B_{22}-B_{21} &        &               &       &
                  & B_{I2}-B_{I1} &        &                \\
                  & B_{12}-B_{12} &        &               &
                  & B_{22}-B_{22} &        &               &       &
                  & B_{I2}-B_{I2} &        &                \\
                  & \vdots        &        &               &
                  & \vdots        &        &               & \dots &
                  & \vdots        &        &                \\
                  & B_{12}-B_{1J} &        &               &
                  & B_{22}-B_{2J} &        &               &       &
                  & B_{I2}-B_{IJ} &        &                \\
                  &               & \ddots &               &
                  &               & \ddots &               &       &
                  &               & \ddots &                \\
                  &               &        & B_{1J}-B_{11} &
                  &               &        & B_{2J}-B_{21} &       &
                  &               &        & B_{IJ}-B_{I1}  \\
                  &               &        & B_{1J}-B_{12} & 
                  &               &        & B_{2J}-B_{22} &       &
                  &               &        & B_{IJ}-B_{I2}  \\
                  &               &        & \vdots        & 
                  &               &        & \vdots        & \dots &
                  &               &        & \vdots         \\
                  &               &        & B_{1J}-B_{1J} & 
                  &               &        & B_{2J}-B_{2J} &       &
                  &               &        & B_{IJ}-B_{IJ}
    \end{smallmatrix} \right].
\end{equation}

As it is, there is no simple way to rewrite this using Kronecker products.
However, if we reorder the lines as $jj' = 11, 21, \dots, J1, 12, \dots$, then we are able to rewrite this matrix as
\begin{equation}
M_B = 1_{\mathcal J} \otimes \big[ (1_{\mathcal I}^\top \otimes \text I_{\mathcal J}) \, \text{diag}(B) \big] -  B^\top \otimes \text I_{\mathcal J}
\end{equation}

where $\text{diag}(B)$ is defined similarly as $\text{diag}(A)$ (also using the row-major order).

Finding a correlated equilibrium is a linear feasibility problem, which we can always reformulate as a linear programming problem:

\begin{align}
\max_{\pi_{ij} \geq 0} ~ & 1_{\mathcal I \mathcal J}^\top \pi \\
\text{s.t.} ~ & 1_{\mathcal I \mathcal J}^\top \pi \leq 1 \\
& M_A \, \pi \geq 0_{\mathcal I^2} \\
& M_B \, \pi \geq 0_{\mathcal J^2}
\end{align}

where $\pi$ is understood as a vector.
We know that the value of this program is 1 (since it is attained for $\pi$ constructed from a Nash equilibrium, for instance), and therefore the constraint $1_{\mathcal I \mathcal J}^\top \pi \leq 1$ must bind.

## Zero-sum games

In zero-sum games, the margins of correlated equilibria constitute Nash equilibrium strategies, or optimal strategies.

**Proposition.** If $(\pi_{ij})$ is a correlated equilibrium, then $p_i = \sum_j \pi_{ij}$ is an optimal strategy for player 1, and $q_j = \sum_i \pi_{ij}$ is an optimal strategy for player 2.

*Proof.*
Recall that in zero-sum games, optimal strategies are solutions to the max-min problem

\begin{equation*}
\max_{p \in \Delta_I} \, \min_{q \in \Delta_J} ~ p^\top A q,
\end{equation*}

and any couple of solutions $(p,q)$ forms a Nash equilibrium.

Here, the correlated equilibrium conditions rewrite as:  
$\sum_j \pi_{ij} A_{ij} \geq \sum_j \pi_{ij} A_{i' j}$ for all $i$, $i'$  
$\sum_i \pi_{ij} A_{ij} \leq \sum_i \pi_{ij} A_{i j'}$ for all $j$, $j'$.

By summing these conditions on $i$ and $j$ respectively, and defining $p_i = \sum_j \pi_{ij}$ and $q_j = \sum_i \pi_{ij}$, we obtain that

\begin{equation*}
\sum_j q_j A_{i' j} \leq \sum_{ij} \pi_{ij} A_{ij} \leq \sum_i p_i A_{i j'} \quad \text{for all $i'$ and $j'$},
\end{equation*}
and as a consequence 
\begin{equation*}
\max_{\tilde p \in \Delta_I} \sum_{ij} \tilde p_i A_{ij} q_j
\leq \sum_{ij} \pi_{ij} A_{ij}
\leq \min_{\tilde q \in \Delta_J} \sum_{ij} p_i A_{ij} \tilde q_j,
\end{equation*}

with equality when $\tilde p = p$ and $\tilde q = q$.
Thus $p$ and $q$ are best responses to each other, so they are optimal strategies. *Q.E.D.*

An immediate consequence is that in a zero-sum game, the payoff of the players at a correlated equilibrium is equal to the value of the game:

\begin{equation*}
\sum_{ij} \pi_{ij} A_{ij}
= \sum_{ij} p_i A_{ij} q_j.
\end{equation*}

We also have the following result:

**Proposition (Forges 1990).**
Let $\pi$ be a correlated equilibrium of a zero-sum game, and let $j$ such that $q_j := \sum_i \pi_{ij} > 0$.
Then the strategy $(\pi_{ij}/q_j)_{i \in \mathcal I}$ is a Nash equilibrium strategy for player 1.

*Proof.*
Let $j$ such that $q_j > 0$ and denote $V$ the value of the game.
For player 1, the expected payoff of playing $(\pi_{ij}/q_j)_{i \in \mathcal I}$ against $q$ is

\begin{equation*}
\sum_{il} \frac{\pi_{ij}}{q_j} q_l A_{il}
= \sum_i \frac{\pi_{ij}}{q_j} \underbrace{\sum_l q_l A_{il}}_{= V \text{ if $p_i > 0$}}
= \sum_i \frac{\pi_{ij}}{q_j} V = V
\qquad \text{since $\pi_{ij} > 0$ implies $p_i > 0$.}
\end{equation*}

(Note that the bracketed equality is true because all pure strategies played with positive probability in a Nash equilibrium have the same expected payoff.)  
Hence $(\pi_{ij}/q_j)_{i \in \mathcal I}$ is a best response to the optimal strategy $q$, and therefore it is itself an optimal strategy. *Q.E.D.*

An immediate consequence of the proposition above is: 

**Corollary.** If a zero-sum game has a unique Nash equilibrium, then it also has a unique correlated equilibrium (which must then be given by $\pi_{ij} = p_i q_j$).

*Proof.*
If $q_j > 0$ we must have $\pi_{ij}/q_j = p_i$ by uniqueness of player 1's optimal strategy, hence $\pi_{ij} = p_i q_j$.  
If $q_j = 0$ then necessarily $\pi_{ij} = 0$ (since $\sum_i \pi_{ij} = q_j$) so we also have $\pi_{ij} = p_i q_j$.
*Q.E.D.*

-------

Here, the correlated equilibrium conditions rewrite as:  
$\sum_j \pi_{ij} A_{ij} \geq \sum_j \pi_{ij} A_{i' j}$ for all $i$, $i'$  
$\sum_i \pi_{ij} A_{ij} \leq \sum_i \pi_{ij} A_{i j'}$ for all $j$, $j'$.

$\max_{\pi_{ij} \geq 0} \min_{\alpha_{ii'}, \beta_{jj'} \geq 0}
~\alpha_{ii'} (\sum_j \pi_{ij} A_{ij} - \sum_j \pi_{ij} A_{i' j})
- \beta_{jj'} (\sum_i \pi_{ij} A_{ij} - \sum_i \pi_{ij} A_{i j'})$

$\sum_j \pi_{ij} > 0 \implies \sum_j \pi_{ij} A_{ij} = V_1$

Recall the linear program associated with finding player 1's Nash strategy in a zero-sum game:

\begin{equation}
V_1 = \max_{ p_i \geq 0, U} ~ U 
\qquad \text{s.t.} ~ U \leq \textstyle\sum_i A_{ij} p_i \quad (\forall j),
\quad \textstyle\sum_i p_i = 1.
\end{equation}

We can introduce new variables $\pi_{ij} \geq 0$ verifying $\sum_j \pi_{ij} = p_i$:

\begin{align}
V_1 &= \max_{ p_i \geq 0, \pi_{ij} \geq 0, U} ~ U 
\qquad \text{s.t.} ~ U \leq \textstyle\sum_i A_{ij} p_i \quad (\forall j),
\quad \textstyle\sum_i p_i = 1,
\quad \textstyle\sum_j \pi_{ij} = p_i \\
&= \max_{ p_i \geq 0, \pi_{ij} \geq 0, U} ~ U 
\qquad \text{s.t.} ~ U \leq \textstyle\sum_i A_{ij'} (\textstyle\sum_j \pi_{ij}) \quad (\forall j'),
\quad \textstyle\sum_{ij} \pi_{ij} = 1,
\quad \textstyle\sum_j \pi_{ij} = p_i \\
&= \max_{ \pi_{ij} \geq 0, U} ~ U 
\qquad \text{s.t.} ~ U \leq \textstyle\sum_j \big(\textstyle\sum_i A_{ij'} \pi_{ij}\big) \quad (\forall j'),
\quad \textstyle\sum_{ij} \pi_{ij} = 1.
\end{align}

We can consider another program, which is more constrained:

\begin{align}
\max_{ \pi_{ij} \geq 0, U} ~ U 
\qquad \text{s.t.} ~ \big(\textstyle\sum_i \pi_{ij'}\big) U \leq \textstyle\sum_i A_{ij'} \pi_{ij} \quad (\forall j, j'),
\quad \textstyle\sum_{ij} \pi_{ij} = 1.
\end{align}

$\min_{\pi_{ij}, U_i, V_j} ~ \sum_i (\sum_j \pi_{ij}) U_i - \sum_j (\sum_i \pi_{ij}) V_j$



\begin{align}
\max_{ \pi_{ij}, U_i, V_j } &~ \sum_i (\sum_j \pi_{ij}) U_i - \sum_j (\sum_i \pi_{ij}) V_j \\
\qquad \text{s.t.} &~ \big(\textstyle\sum_j \pi_{ij}\big) U_i \geq \textstyle\sum_j \pi_{ij} A_{ij'} \quad (\forall i, i'),
\qquad \text{s.t.} &~ \big(\textstyle\sum_i \pi_{ij}\big) V_j \leq \textstyle\sum_i \pi_{ij} A_{ij'} \quad (\forall j, j'),
\quad \textstyle\sum_{ij} \pi_{ij} = 1.
\end{align}

--------

In zero-sum games, the margins of correlated equilibria coincide with Nash equilibria, in the sense that if $(\pi_{ij})$ is a correlated equilibrium, then $p_i = \sum_j \pi_{ij}$ and $q_j = \sum_i \pi_{ij}$ constitute a Nash equilibrium.
An immediate consequence is that in a zero-sum game, the payoff of the players at a correlated equilibrium is equal to the value of the game.

Recall the conditions for $(p,q)$ to be a Nash equilibrium:

$\sum_{ij} p_i A_{ij} q_j \geq \sum_j A_{i'j} q_j$ for all $i'$

$\sum_{ij} p_i B_{ij} q_j \geq \sum_j B_{ij'} q_j$ for all $j'$.

Let's show that the first condition holds.
Fix $i'$, then for all $j$ we have

\begin{align}
\sum_i p_i A_{ij}
&= \sum_{il} A_{ij} \pi_{il} \\
&\geq \sum_{il} A_{il} \pi_{il}
\quad \text{since } \sum_i B_{il} \pi_{il} \geq \sum_i B_{ij} \pi_{il} \text{ and } B_{ij} = -A_{ij} \\
&\geq \sum_{il} A_{i' l} \pi_{il}
\quad \text{since } \sum_l A_{il} \pi_{il} \geq \sum_l A_{i' l} \pi_{il} \\
&= \sum_l A_{i' l} q_l
\end{align}
hence
\begin{equation}
\sum_{ij} p_i A_{ij} q_j
= \sum_j \left( \sum_i p_i A_{ij} \right) q_j
\geq \left( \sum_l A_{i' l} q_l \right) \sum_j q_j
= \sum_l A_{i' l} q_l.
\end{equation}

The second condition holds as well by symmetry.

**Question.** If we have a Nash $(p,q)$, does any $\pi$ such that its margins are $p$ and $q$ constitute a correlated equilibrium?

**Proposition (Forges 1990).**
Let $\pi$ be a correlated equilibrium of a zero-sum game, and let $j$ such that $q_j := \sum_i \pi_{ij} > 0$.
Then the strategy $(\pi_{ij}/q_j)_{i \in \mathcal I}$ is a Nash equilibrium strategy for player 1.

An immediate consequence of the proposition above is that, if a zero-sum game has a unique Nash equilibrium, then it also has a unique correlated equilibrium (which must then be $\pi_{ij} = p_i q_j$).
Indeed, if $q_j > 0$ we must have $\pi_{ij}/q_j = p_i$ by uniqueness of player 1's Nash equilibrium strategy, hence $\pi_{ij} = p_i q_j$.
If $q_j = 0$ then $\pi_{ij} = 0$ since $\sum_i \pi_{ij} = q_j$ so we also have $\pi_{ij} = p_i q_j$.

# Computing correlated equilibria

## Computation using Gurobi

In [5]:
def Bimatrix_game_solve_AumannEq(self, verbose=0):
    model = grb.Model()
    model.Params.OutputFlag = 0
    π = model.addMVar(shape=(self.nbi,self.nbj))
    model.setObjective(π.sum(), grb.GRB.MAXIMIZE)
    model.addConstr(π.sum() <= 1)
    for i in range(self.nbi) :
        for k in range(self.nbi) :
            if k != i : model.addConstr(self.A_i_j[i,:] @ π[i,:] >= self.A_i_j[k,:] @ π[i,:])
    for j in range(self.nbj) :
        for l in range(self.nbj) :
            if l != j : model.addConstr(self.B_i_j[:,j] @ π[:,j] >= self.B_i_j[:,l] @ π[:,j])
    model.optimize() 
    π_i_j = np.array(model.getAttr('x')).reshape(self.nbi,self.nbj)
    if verbose > 0: print('π_i_j =\n', π_i_j)
    return {'π_i_j': π_i_j,
            'val1': self.A_i_j.flatten() @ π_i_j.flatten(),
            'val2': self.B_i_j.flatten() @ π_i_j.flatten()}

Bimatrix_game.solve_AumannEq = Bimatrix_game_solve_AumannEq

In [6]:
sol = penalty_nonzero_sum.solve_AumannEq(verbose=1)
π_i_j, val1, val2 = sol['π_i_j'], sol['val1'], sol['val2']

sol

Set parameter Username
Academic license - for non-commercial use only - expires 2025-12-03
π_i_j =
 [[0.0739268  0.03428835 0.22921855]
 [0.0545915  0.02532035 0.16926722]
 [0.09056711 0.04200638 0.28081374]]


{'π_i_j': array([[0.0739268 , 0.03428835, 0.22921855],
        [0.0545915 , 0.02532035, 0.16926722],
        [0.09056711, 0.04200638, 0.28081374]]),
 'val1': 82.62606457928054,
 'val2': 36.37698012002112}

In [7]:
penalty_nonzero_sum.is_AumannEq(π_i_j)

True

In [8]:
penalty_nonzero_sum.is_NashEq(π_i_j.sum(axis=1), π_i_j.sum(axis=0))

True

In [9]:
np.outer(π_i_j.sum(axis=1), π_i_j.sum(axis=0))

array([[0.0739268 , 0.03428835, 0.22921855],
       [0.0545915 , 0.02532035, 0.16926722],
       [0.09056711, 0.04200638, 0.28081374]])

## Computation using the simplex

Recall the three inequalities we need to implement:

$1_{\mathcal{IJ}}^\top \, \pi \leq 1$

$\big( \big[ (\text I_{\mathcal I} \otimes 1_{\mathcal J}^\top) \, \text{diag}(A) \big] \otimes 1_{\mathcal I} - \text I_{\mathcal I} \otimes A\big) \, \pi \geq 0$

$\big( 1_{\mathcal J} \otimes \big[ (1_{\mathcal I}^\top \otimes \text I_{\mathcal J}) \, \text{diag}(B) \big] -  B^\top \otimes \text I_{\mathcal J} \big) \, \pi \geq 0$.

The associated linear program is degenerate (because some inequalities have a constant 0 on the right-hand side) so we solve an approximation with the simplex.

In [10]:
from mec.lp import Tableau

def Bimatrix_game_simplex_solve_AumannEq(self, verbose=0, eps=1e-5):
    M_1 = np.ones(self.nbi*self.nbj)
    M_A = np.kron( np.kron(np.eye(self.nbi), np.ones((1, self.nbj))) @ np.diag(self.A_i_j.flatten()),
                   np.ones((self.nbi, 1))
                 ) - np.kron(np.eye(self.nbi), self.A_i_j)
    M_B = np.kron( np.ones((self.nbj, 1)),
                   np.kron(np.ones((1, self.nbi)), np.eye(self.nbj)) @ np.diag(self.B_i_j.flatten())
                 ) - np.kron(self.B_i_j.T, np.eye(self.nbj))
    tab = Tableau(A_i_j = np.vstack([M_1, -M_A, -M_B]),
                  b_i = np.eye(1+self.nbi**2+self.nbj**2)[0] + eps*(1-np.eye(1+self.nbi**2+self.nbj**2)[0]),
                  c_j = np.ones(self.nbi*self.nbj),
                  decision_var_names_j = ['π_'+str(i)+'_'+str(j) for j in range(self.nbj) for i in range(self.nbi)])
    πstar, _, _ = tab.simplex_solve()
    π_i_j = np.array(πstar.reshape(self.nbi,self.nbj))
    if verbose > 0: print('π_i_j = \n', π_i_j)
    return {'π_i_j': π_i_j,
            'val1': self.A_i_j.flatten() @ π_i_j.flatten(),
            'val2': self.B_i_j.flatten() @ π_i_j.flatten()}

Bimatrix_game.simplex_solve_AumannEq = Bimatrix_game_simplex_solve_AumannEq

In [11]:
sol = penalty_nonzero_sum.simplex_solve_AumannEq(verbose=1, eps=1e-5)
π_i_j, val1, val2 = sol['π_i_j'], sol['val1'], sol['val2']

sol

π_i_j = 
 [[0.07392655 0.03428756 0.22921896]
 [0.05459142 0.02532056 0.16926736]
 [0.09056694 0.04200653 0.28081412]]


{'π_i_j': array([[0.07392655, 0.03428756, 0.22921896],
        [0.05459142, 0.02532056, 0.16926736],
        [0.09056694, 0.04200653, 0.28081412]]),
 'val1': 82.62607459750402,
 'val2': 36.37698676485681}

In [12]:
penalty_nonzero_sum.is_AumannEq(π_i_j)

Player 1 can deviate from i = 0 with k = 1.


False

In [13]:
penalty_nonzero_sum.is_NashEq(π_i_j.sum(axis=1), π_i_j.sum(axis=0))

Pure strategy i = 0 beats p_i.


False

## Benchmarking

In [14]:
I, J = 10, 7
random_game = Bimatrix_game(A_i_j = 100*np.random.rand(I,J),
                            B_i_j = 100*np.random.rand(I,J))

### Simplex

In [15]:
sol = random_game.simplex_solve_AumannEq(eps=1e-8)
π_i_j, val1, val2 = sol['π_i_j'], sol['val1'], sol['val2']

In [16]:
random_game.is_AumannEq(π_i_j)

True

In [17]:
random_game.is_NashEq(π_i_j.sum(axis=1), π_i_j.sum(axis=0))

Pure strategy i = 4 beats p_i.


False

### Gurobi

In [18]:
sol = random_game.solve_AumannEq()
π_i_j, val1, val2 = sol['π_i_j'], sol['val1'], sol['val2']

In [19]:
random_game.is_AumannEq(π_i_j)

True

In [20]:
random_game.is_NashEq(π_i_j.sum(axis=1), π_i_j.sum(axis=0))

Pure strategy i = 2 beats p_i.


False